In [1]:
import os
import fitz  import numpy as np
import hashlib
import pickle
import httpx
from typing import List
from openai import OpenAI
from chromadb import PersistentClient
from chromadb.api.types import Documents, Embeddings, EmbeddingFunction


In [6]:
from dotenv import load_dotenv

load_dotenv()
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")

In [7]:
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50
EMBEDDING_MODEL = "text-embedding-ada-002"
CHAT_MODEL = "gpt-3.5-turbo"
CHROMA_PATH = ".chroma"
COLLECTION_NAME = "pdf_chunks_openai"
CACHE_DIR = ".cache"
os.makedirs(CACHE_DIR, exist_ok=True)

class CustomHTTPClient(httpx.Client):
    def __init__(self, *args, **kwargs):
        kwargs.pop("proxies", None)
        super().__init__(*args, **kwargs)

client = OpenAI(http_client=CustomHTTPClient())


In [8]:
class OpenAIEmbeddingFunction(EmbeddingFunction[Documents]):
    def __call__(self, input: Documents) -> Embeddings:
        embeddings = []
        BATCH_SIZE = 100
        for i in range(0, len(input), BATCH_SIZE):
            batch = input[i:i + BATCH_SIZE]
            response = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
            embeddings.extend([e.embedding for e in response.data])
        return embeddings


In [9]:
def extract_text_from_pdf(path: str) -> str:
    doc = fitz.open(path)
    return "\n".join([page.get_text() for page in doc])

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    words = text.split()
    return [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size - overlap)]

def get_file_hash(filepaths: List[str]) -> str:
    m = hashlib.md5()
    for path in sorted(filepaths):
        with open(path, 'rb') as f:
            m.update(f.read())
    return m.hexdigest()


In [10]:
file_paths = [os.path.join("data", f) for f in os.listdir("data") if f.endswith(".pdf")]
all_chunks = []
for pdf_path in file_paths:
    text = extract_text_from_pdf(pdf_path)
    chunks = chunk_text(text)
    all_chunks.extend(chunks)


In [11]:
pdf_hash = get_file_hash(file_paths)
cache_file = os.path.join(CACHE_DIR, f"{pdf_hash}.pkl")
is_cache_hit = os.path.exists(cache_file)
ids = [f"chunk_{i}" for i in range(len(all_chunks))]

if is_cache_hit:
    with open(cache_file, "rb") as f:
        cached_ids, cached_chunks = pickle.load(f)
else:
    cached_ids = ids
    cached_chunks = all_chunks
    with open(cache_file, "wb") as f:
        pickle.dump((cached_ids, cached_chunks), f)


In [12]:
chroma_client = PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME, embedding_function=OpenAIEmbeddingFunction())

if len(collection.get()["ids"]) == 0:
    collection.add(documents=cached_chunks, ids=cached_ids)


/var/folders/69/fwpfz67x4szfdnkld_9kl1g80000gp/T/ipykernel_51940/3413593036.py:2: DeprecationWarning: The class OpenAIEmbeddingFunction does not implement __init__. This will be required in a future version.
  collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME, embedding_function=OpenAIEmbeddingFunction())


In [13]:
def generate_answer(context: str, query: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are an AI assistant. Answer only using the provided context. "
                "If the answer is not in the context, say 'I don't know, I couldn't find the relevant answer'."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
        }
    ]
    response = client.chat.completions.create(model=CHAT_MODEL, messages=messages)
    return response.choices[0].message.content.strip()

query = input("Enter your question: ")
results = collection.query(query_texts=[query], n_results=3)
top_chunks = results['documents'][0]
context = "\n".join(top_chunks)
answer = generate_answer(context, query)

print(" Context Used:")
print(context)
print("\n Answer:")
print(answer)


Context Used:
steps. It can be used for business intelligence, analytics, and machine learning applications and helps organizations scale to collect, process, and store data. Fundamentals of Data Engineering Concepts (ISBN: 978-93-48620-19-4) 19 A modern data pipeline contains multiple consecutive phases like data ingestion, transformation, validation, storage, and monitoring, etc. By providing a unified view of enterprise data across disparate sources, data integration simplifies the process of combining data for analysis. 4.5.1 Data pipeline hierarchy: The hierarchy clarifies the distinct stages that data traverses, ending in informed decision-making. Data pipeline staircase diagram data flows. In every step the previous one is the cornerstone of data raw input that gives valuable insights. Here are further details on every step in the hierarchy (as shown in fig.4.7): Fig. 4.7: Data Pipeline Hierarchy a. Data Collection Data collection forms the basis of the data pipeline where raw d